# Pyright benchmark history

Explore checked-in Linux x64 benchmark results across Pyright commits, compare the latest run with a baseline, and optionally export charts and a static GitHub Pages dashboard.

The workflow adds commit metadata when collecting hosted results. Notebook enrichment is dry-run by default; set `WRITE_ENRICHED_RESULTS = True` only for result files whose source commit is the current checkout.

In [ ]:
# 1. Import analysis and visualization libraries
from __future__ import annotations

import html
import json
import os
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# 2. Define benchmark file and metadata schemas
REQUIRED_METADATA = {
    "timestamp": str,
    "source_revision": str,
    "source_commit_subject": str,
    "source_commit_timestamp": str,
    "results": list,
}

METRIC_SPECS = {
    "execution_time_s": {"label": "Execution time", "unit": "seconds", "lower_is_better": True},
    "peak_memory_mb": {"label": "Peak memory", "unit": "MiB", "lower_is_better": True},
}


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "build" / "benchmark" / "baselines").is_dir():
            return candidate
    raise FileNotFoundError("Could not find build/benchmark/baselines from the current directory")


REPO_ROOT = find_repo_root(Path.cwd())
BASELINE_DIR = REPO_ROOT / "build" / "benchmark" / "baselines"
EXPORT_DIR = Path(
    os.environ.get("PYRIGHT_BENCHMARK_EXPORT_DIR", REPO_ROOT / "docs" / "benchmark-results")
).resolve()
WRITE_ENRICHED_RESULTS = os.environ.get("PYRIGHT_BENCHMARK_ENRICH", "") == "1"
EXPORT_ASSETS = os.environ.get("PYRIGHT_BENCHMARK_EXPORT", "") == "1"

print(f"Repository: {REPO_ROOT}")
print(f"Baseline directory: {BASELINE_DIR}")

In [ ]:
# 3. Capture Git commit name and timestamp
def git_value(*args: str) -> str:
    try:
        return subprocess.run(
            ["git", "-C", str(REPO_ROOT), *args],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as error:
        raise RuntimeError("Git history is unavailable; run this notebook from a Git checkout") from error


def current_commit_metadata() -> dict[str, str]:
    revision = git_value("rev-parse", "HEAD")
    return {
        "source_revision": revision,
        "source_commit_short": git_value("rev-parse", "--short=12", revision),
        "source_commit_subject": git_value("show", "-s", "--format=%s", revision),
        "source_commit_timestamp": git_value("show", "-s", "--format=%cI", revision),
    }


current_commit = current_commit_metadata()
display(current_commit)

In [ ]:
# 4. Attach commit metadata to result files
def enrich_result_file(
    path: Path,
    commit: dict[str, str],
    *,
    write: bool = False,
) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"{path} must contain a JSON object")

    metadata = {key: value for key, value in commit.items() if key != "source_commit_short"}
    conflicts = {
        key: data[key]
        for key, value in metadata.items()
        if key in data and data[key] != value
    }
    if conflicts:
        raise ValueError(f"Refusing to replace existing commit metadata in {path}: {conflicts}")
    enriched = {**metadata, **data}
    if "timestamp" not in enriched:
        enriched["timestamp"] = datetime.now().astimezone().isoformat()
    if write:
        path.write_text(json.dumps(enriched, indent=2) + "\n", encoding="utf-8")
    return enriched


result_files = sorted(BASELINE_DIR.glob("benchmark_*_linux-x64.json"))
RESULT_FILES_TO_ENRICH: list[Path] = []
if WRITE_ENRICHED_RESULTS:
    if not RESULT_FILES_TO_ENRICH:
        raise RuntimeError("Set RESULT_FILES_TO_ENRICH explicitly before enabling enrichment")
    for result_file in RESULT_FILES_TO_ENRICH:
        enrich_result_file(result_file, current_commit, write=True)
    print(f"Enriched {len(RESULT_FILES_TO_ENRICH)} result file(s)")
else:
    print(f"Dry run: {len(result_files)} result file(s) discovered; no files changed")

In [ ]:
# 5. Load and validate historical benchmark results
def parse_iso8601(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def validate_result(path: Path, data: Any) -> list[str]:
    problems: list[str] = []
    if not isinstance(data, dict):
        return ["root value is not an object"]
    for field, expected_type in REQUIRED_METADATA.items():
        if not isinstance(data.get(field), expected_type):
            problems.append(f"{field} must be {expected_type.__name__}")
    for field in ("timestamp", "source_commit_timestamp"):
        try:
            parse_iso8601(str(data.get(field, "")))
        except ValueError:
            problems.append(f"{field} must be an ISO 8601 timestamp")
    if not isinstance(data.get("source_revision"), str) or len(data.get("source_revision", "")) != 40:
        problems.append("source_revision must be a 40-character SHA")
    return problems


def load_history(paths: list[Path]) -> tuple[list[dict[str, Any]], list[dict[str, str]]]:
    records: list[dict[str, Any]] = []
    failures: list[dict[str, str]] = []
    for path in paths:
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            problems = validate_result(path, data)
            if problems:
                failures.append({"file": path.name, "error": "; ".join(problems)})
                continue
            for package in data["results"]:
                metrics = package.get("metrics", {}).get("pyright", {})
                if not metrics.get("ok"):
                    continue
                for metric, spec in METRIC_SPECS.items():
                    value = metrics.get(metric)
                    if isinstance(value, (int, float)):
                        records.append(
                            {
                                "file": path.name,
                                "benchmark": package["package_name"],
                                "metric": metric,
                                "value": value,
                                "unit": spec["unit"],
                                "lower_is_better": spec["lower_is_better"],
                                "commit_sha": data["source_revision"],
                                "commit_title": data["source_commit_subject"],
                                "commit_timestamp": data["source_commit_timestamp"],
                                "collection_timestamp": data["timestamp"],
                            }
                        )
        except (OSError, ValueError, KeyError, TypeError) as error:
            failures.append({"file": path.name, "error": str(error)})
    return records, failures


records, malformed_files = load_history(result_files)
print(f"Loaded {len(records)} measurements from {len(result_files)} result file(s)")
if malformed_files:
    display(pd.DataFrame(malformed_files))

In [ ]:
# 6. Normalize results into a DataFrame
history = pd.DataFrame.from_records(records)
if history.empty:
    raise RuntimeError("No valid benchmark measurements were found")

for column in ("commit_timestamp", "collection_timestamp"):
    history[column] = pd.to_datetime(history[column], utc=True, errors="raise")
history["value"] = pd.to_numeric(history["value"], errors="raise")
history["short_sha"] = history["commit_sha"].str[:12]
history = (
    history.sort_values(["collection_timestamp", "benchmark", "metric"])
    .drop_duplicates(["commit_sha", "benchmark", "metric"], keep="last")
    .reset_index(drop=True)
)

run_summary = (
    history.groupby(
        ["collection_timestamp", "commit_sha", "short_sha", "commit_title", "commit_timestamp"],
        as_index=False,
    )
    .agg(
        package_count=("benchmark", "nunique"),
        total_execution_time_s=("value", lambda values: history.loc[values.index].query("metric == 'execution_time_s'")["value"].sum()),
        max_peak_memory_mb=("value", lambda values: history.loc[values.index].query("metric == 'peak_memory_mb'")["value"].max()),
    )
)
display(run_summary)

In [ ]:
# 7. Graph performance across commits
SELECTED_PACKAGES = sorted(history["benchmark"].unique())
figures: dict[str, plt.Figure] = {}

for metric, spec in METRIC_SPECS.items():
    metric_data = history[
        (history["metric"] == metric) & history["benchmark"].isin(SELECTED_PACKAGES)
    ]
    figure, axis = plt.subplots()
    sns.lineplot(
        data=metric_data,
        x="collection_timestamp",
        y="value",
        hue="benchmark",
        marker="o",
        errorbar=None,
        palette="colorblind",
        ax=axis,
    )
    for row in metric_data.itertuples():
        axis.annotate(
            row.short_sha,
            (row.collection_timestamp, row.value),
            xytext=(4, 5),
            textcoords="offset points",
            fontsize=7,
            alpha=0.8,
        )
    axis.set_title(f"Pyright {spec['label'].lower()} across commits")
    axis.set_xlabel("Benchmark collection time (UTC)")
    axis.set_ylabel(f"{spec['label']} ({spec['unit']})")
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d", tz=history["collection_timestamp"].dt.tz))
    axis.legend(title="Package", bbox_to_anchor=(1.02, 1), loc="upper left")
    figure.autofmt_xdate()
    figure.tight_layout()
    figures[metric] = figure
    plt.show()

In [ ]:
# 8. Compare the latest commit with the baseline
BASELINE_REVISION: str | None = None


def compare_latest(data: pd.DataFrame, baseline_revision: str | None = None) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for (benchmark, metric), group in data.groupby(["benchmark", "metric"], sort=True):
        ordered = group.sort_values("collection_timestamp")
        latest = ordered.iloc[-1]
        if baseline_revision:
            matches = ordered[ordered["commit_sha"].str.startswith(baseline_revision)]
            if matches.empty:
                continue
            baseline = matches.iloc[-1]
        else:
            baseline = ordered.iloc[0]
        absolute_change = latest["value"] - baseline["value"]
        percent_change = absolute_change / baseline["value"] * 100 if baseline["value"] else float("nan")
        lower_is_better = bool(latest["lower_is_better"])
        if abs(percent_change) < 1:
            assessment = "Stable"
        elif (percent_change < 0) == lower_is_better:
            assessment = "Improvement"
        else:
            assessment = "Regression"
        rows.append(
            {
                "benchmark": benchmark,
                "metric": metric,
                "unit": latest["unit"],
                "baseline_sha": baseline["short_sha"],
                "latest_sha": latest["short_sha"],
                "baseline": baseline["value"],
                "latest": latest["value"],
                "absolute_change": absolute_change,
                "percent_change": percent_change,
                "assessment": assessment,
            }
        )
    return pd.DataFrame(rows)


comparison = compare_latest(history, BASELINE_REVISION)
display(
    comparison.style.format(
        {"baseline": "{:.2f}", "latest": "{:.2f}", "absolute_change": "{:+.2f}", "percent_change": "{:+.1f}%"}
    ).map(
        lambda value: "color: #087f5b; font-weight: bold" if value == "Improvement" else "color: #c92a2a; font-weight: bold" if value == "Regression" else "",
        subset=["assessment"],
    )
)

In [ ]:
# 9. Export charts and summary data
def export_assets(
    output_dir: Path,
    data: pd.DataFrame,
    summary: pd.DataFrame,
    charts: dict[str, plt.Figure],
) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs = {
        "history_csv": output_dir / "history.csv",
        "comparison_json": output_dir / "comparison.json",
    }
    data.to_csv(outputs["history_csv"], index=False)
    outputs["comparison_json"].write_text(
        json.dumps(summary.to_dict(orient="records"), indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    for metric, figure in charts.items():
        chart_path = output_dir / f"{metric}.svg"
        figure.savefig(chart_path, format="svg", bbox_inches="tight")
        outputs[metric] = chart_path
    return outputs


exported: dict[str, Path] = {}
if EXPORT_ASSETS:
    exported = export_assets(EXPORT_DIR, history, comparison, figures)
    display(exported)
else:
    print("Exports disabled. Set EXPORT_ASSETS = True in Section 2 to write dashboard assets.")

In [ ]:
# 10. Generate a GitHub Pages dashboard
def dashboard_html(run_data: pd.DataFrame, summary: pd.DataFrame) -> str:
    latest = run_data.sort_values("collection_timestamp").iloc[-1]
    status_class = {"Improvement": "good", "Regression": "bad", "Stable": "stable"}
    rows = "".join(
        "<tr>"
        f"<td>{html.escape(str(row.benchmark))}</td>"
        f"<td>{html.escape(METRIC_SPECS[str(row.metric)]['label'])}</td>"
        f"<td>{float(row.baseline):.2f} {html.escape(str(row.unit))}</td>"
        f"<td>{float(row.latest):.2f} {html.escape(str(row.unit))}</td>"
        f"<td>{float(row.percent_change):+.1f}%</td>"
        f"<td class='{status_class[str(row.assessment)]}'>{html.escape(str(row.assessment))}</td>"
        "</tr>"
        for row in summary.itertuples()
    )
    charts = "".join(
        f"<section><h2>{html.escape(spec['label'])}</h2><img src='{metric}.svg' alt='{html.escape(spec['label'])} trend chart'></section>"
        for metric, spec in METRIC_SPECS.items()
    )
    return f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Pyright benchmark history</title><style>
:root{{--ink:#18222c;--paper:#f4f7f5;--line:#cdd8d3;--good:#087f5b;--bad:#c92a2a;--stable:#495057}}
body{{margin:0;background:var(--paper);color:var(--ink);font-family:Segoe UI,sans-serif}}header,main{{max-width:1180px;margin:auto;padding:28px}}header{{border-bottom:4px solid #e67700}}h1,h2{{font-family:Georgia,serif}}section{{margin:32px 0}}img{{max-width:100%;height:auto;background:white;border:1px solid var(--line)}}table{{width:100%;border-collapse:collapse;background:white}}th,td{{padding:10px;border:1px solid var(--line);text-align:right}}th:first-child,td:first-child{{text-align:left}}.good{{color:var(--good)}}.bad{{color:var(--bad)}}.stable{{color:var(--stable)}}code{{overflow-wrap:anywhere}}
</style></head><body><header><h1>Pyright benchmark history</h1>
<p>Latest: <code>{html.escape(str(latest['commit_sha']))}</code> {html.escape(str(latest['commit_title']))}</p>
<p>Collected {html.escape(str(latest['collection_timestamp']))}</p></header><main>{charts}
<section><h2>Latest compared with baseline</h2><table><thead><tr><th>Package</th><th>Metric</th><th>Baseline</th><th>Latest</th><th>Change</th><th>Assessment</th></tr></thead><tbody>{rows}</tbody></table></section>
</main></body></html>"""


dashboard = dashboard_html(history, comparison)
if EXPORT_ASSETS:
    dashboard_path = EXPORT_DIR / "index.html"
    dashboard_path.write_text(dashboard, encoding="utf-8")
    print(f"Dashboard: {dashboard_path}")
else:
    display(HTML(dashboard))

In [ ]:
# 11. Automate notebook and page generation
NOTEBOOK_PATH = REPO_ROOT / "build" / "benchmark" / "benchmark_history.ipynb"
EXECUTED_NOTEBOOK = EXPORT_DIR / "benchmark_history.executed.ipynb"
NBCONVERT_COMMAND = [
    "jupyter",
    "nbconvert",
    "--to",
    "notebook",
    "--execute",
    str(NOTEBOOK_PATH),
    "--output",
    EXECUTED_NOTEBOOK.name,
    "--output-dir",
    str(EXPORT_DIR),
    "--ExecutePreprocessor.timeout=180",
]

assert not malformed_files, f"Malformed benchmark files: {malformed_files}"
assert not history.empty, "No normalized benchmark data"
assert set(METRIC_SPECS).issubset(set(history["metric"])), "Required metrics are missing"

print("For CI, execute with PYRIGHT_BENCHMARK_EXPORT=1:")
print(subprocess.list2cmdline(NBCONVERT_COMMAND))
print(f"Expected dashboard: {EXPORT_DIR / 'index.html'}")
print("Fail CI if notebook execution fails, malformed_files is non-empty, or index.html is absent.")